## Обработка pkl данных в parquet

In [ ]:



import pickle
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


INPUT_PATH = Path("output/spreads_1.pkl")
PARQUET_DIR = Path("output/spreads_parquet_v2_1")
PARQUET_DIR.mkdir(parents=True, exist_ok=True)

NUM_COLS = [
    "spread_long",
    "spread_short",
    "okx_latency_ms",
    "bybit_latency_ms",
    "calc_local_ts_ms",
    "okx_local_recv_ts_ms",
    "okx_ts_ms",
    "bybit_local_recv_ts_ms",
    "bybit_ts_ms",
]


def normalize_batch(batch):
    df = pd.DataFrame(batch)
    if df.empty:
        return df

    for col in NUM_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df["okx_freshness_ms"] = df["calc_local_ts_ms"] - df["okx_local_recv_ts_ms"]
    df["bybit_freshness_ms"] = df["calc_local_ts_ms"] - df["bybit_local_recv_ts_ms"]

    df["event_local_ts_ms"] = df["okx_local_recv_ts_ms"]
    mask_bybit = df["trigger"].eq("bybit")
    df.loc[mask_bybit, "event_local_ts_ms"] = df.loc[mask_bybit, "bybit_local_recv_ts_ms"]

    df["event_dt"] = pd.to_datetime(df["event_local_ts_ms"], unit="ms", errors="coerce")
    df = df[df["event_dt"].notna()].copy()

    df["event_date"] = df["event_dt"].dt.strftime("%Y-%m-%d")
    df["event_hour"] = df["event_dt"].dt.strftime("%H")
    df["max_freshness_ms"] = df[["okx_freshness_ms", "bybit_freshness_ms"]].max(axis=1)
    df["max_latency_ms"] = df[["okx_latency_ms", "bybit_latency_ms"]].max(axis=1)

    keep_cols = [
        "event_dt",
        "event_local_ts_ms",
        "event_date",
        "event_hour",
        "base_coin",
        "trigger",
        "spread_long",
        "spread_short",
        "okx_latency_ms",
        "bybit_latency_ms",
        "okx_freshness_ms",
        "bybit_freshness_ms",
        "max_freshness_ms",
        "max_latency_ms",
        "calc_local_ts_ms",
        "okx_local_recv_ts_ms",
        "okx_ts_ms",
        "bybit_local_recv_ts_ms",
        "bybit_ts_ms",
    ]
    keep_cols = [c for c in keep_cols if c in df.columns]
    return df[keep_cols].copy()


def write_partitioned_files(df: pd.DataFrame, batch_idx: int):
    if df.empty:
        return 0

    written = 0

    for (event_date, event_hour), sub in df.groupby(["event_date", "event_hour"], sort=False):
        part_dir = PARQUET_DIR / f"event_date={event_date}" / f"event_hour={event_hour}"
        part_dir.mkdir(parents=True, exist_ok=True)

        file_path = part_dir / f"batch_{batch_idx:06d}.parquet"
        table = pa.Table.from_pandas(sub.drop(columns=["event_date", "event_hour"]), preserve_index=False)
        pq.write_table(table, file_path, compression="zstd")

        written += len(sub)

    return written


batch_idx = 0
total_rows = 0

with INPUT_PATH.open("rb") as fh:
    while True:
        try:
            batch = pickle.load(fh)
        except EOFError:
            print("EOF")
            break
        except Exception as e:
            print(f"Stop on broken tail at batch_idx={batch_idx}: {e}")
            break

        if not isinstance(batch, list) or not batch:
            batch_idx += 1
            continue

        df = normalize_batch(batch)
        if df.empty:
            batch_idx += 1
            continue

        written = write_partitioned_files(df, batch_idx)
        total_rows += written

        print(
            f"batch={batch_idx} rows={len(df)} written={written} "
            f"min_dt={df['event_dt'].min()} max_dt={df['event_dt'].max()}"
        )
        batch_idx += 1

print("TOTAL_ROWS_WRITTEN:", total_rows)

## Перевод паркет в дб и плот 3д+2д графиков 

In [ ]:


import duckdb
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from pathlib import Path
from plotly.offline import plot

pio.renderers.default = "browser"

PARQUET_GLOB = "output/spreads_parquet_v2_1/**/*.parquet"


def load_plot_df(
    start_dt: str,
    end_dt: str,
    spread_col: str = "spread_long",
    coin_offset: int = 0,
    top_n_coins: int = 30,
    max_rows_per_coin:int=500
) -> pd.DataFrame:
    con = duckdb.connect()

    extra_filters = []

    extra_filter_sql = ""
    if extra_filters:
        extra_filter_sql = " AND " + " AND ".join(extra_filters)

    query_sql = f"""
        WITH base_data AS (
            SELECT
                event_dt,
                base_coin,
                trigger,
                {spread_col} AS spread_value,
                okx_latency_ms,
                bybit_latency_ms,
                okx_freshness_ms,
                bybit_freshness_ms,
                max_latency_ms,
                max_freshness_ms
            FROM read_parquet('{PARQUET_GLOB}')
            WHERE event_dt >= TIMESTAMP '{start_dt}'
              AND event_dt < TIMESTAMP '{end_dt}'
              AND base_coin IS NOT NULL
              AND {spread_col} IS NOT NULL
        ),
        coin_counts AS (
            SELECT
                base_coin,
                COUNT(*) AS n_rows
            FROM base_data
            GROUP BY base_coin
        ),
        ranked_coins AS (
            SELECT
                base_coin,
                n_rows,
                ROW_NUMBER() OVER (ORDER BY n_rows DESC, base_coin ASC) AS coin_rank
            FROM coin_counts
        ),
        selected_coins AS (
            SELECT
                base_coin,
                coin_rank
            FROM ranked_coins
            WHERE coin_rank > {coin_offset}
              AND coin_rank <= {coin_offset + top_n_coins}
        )
        SELECT
            s.event_dt,
            s.base_coin,
            sc.coin_rank,
            s.trigger,
            s.spread_value,
            s.okx_latency_ms,
            s.bybit_latency_ms,
            s.okx_freshness_ms,
            s.bybit_freshness_ms,
            s.max_latency_ms,
            s.max_freshness_ms
        FROM base_data AS s
        INNER JOIN selected_coins AS sc
            ON s.base_coin = sc.base_coin
        WHERE 1=1
            {extra_filter_sql}
        ORDER BY sc.coin_rank, s.event_dt
    """

    df = con.execute(query_sql).df()

    if df.empty:
        return df

    df["event_dt"] = pd.to_datetime(df["event_dt"])
    df["coin_rank"] = df["coin_rank"].astype(int)
    df["legend_name"] = df["coin_rank"].astype(str) + ". " + df["base_coin"].astype(str)

    if max_rows_per_coin is not None:
        sampled_parts = []
        for _, sub in df.groupby("base_coin", sort=False):
            sub = sub.sort_values("event_dt").copy()
            if len(sub) > max_rows_per_coin:
                idx = np.linspace(0, len(sub) - 1, num=max_rows_per_coin, dtype=int)
                sub = sub.iloc[idx].copy()
            sampled_parts.append(sub)
        df = pd.concat(sampled_parts, ignore_index=True)

    df = df.sort_values(["coin_rank", "event_dt"]).reset_index(drop=True)
    return df

def inspect_plot_df(df: pd.DataFrame) -> None:
    if df.empty:
        print("plot_df is empty")
        return

    print("shape:", df.shape)
    print("time min:", df["event_dt"].min())
    print("time max:", df["event_dt"].max())
    print("duration sec:", (df["event_dt"].max() - df["event_dt"].min()).total_seconds())

    summary = (
        df.groupby("base_coin")
        .agg(
            n=("base_coin", "size"),
            tmin=("event_dt", "min"),
            tmax=("event_dt", "max"),
            spread_min=("spread_value", "min"),
            spread_max=("spread_value", "max"),
        )
        .sort_values("n", ascending=False)
        .head(20)
    )
    print(summary)

def plot_spread_2d(df: pd.DataFrame) -> None:
    if df.empty:
        raise ValueError("plot_df is empty")

    fig = go.Figure()

    for _, sub in df.groupby("coin_rank", sort=True):
        sub = sub.sort_values("event_dt")
        legend_name = sub["legend_name"].iloc[0]

        fig.add_trace(
            go.Scatter(
                x=sub["event_dt"],
                y=sub["spread_value"],
                mode="lines+markers",
                name=legend_name,
                line=dict(width=1.7),
                marker=dict(size=3),
                customdata=np.stack([
                    sub["trigger"].astype(str),
                    sub["okx_latency_ms"].fillna(np.nan),
                    sub["bybit_latency_ms"].fillna(np.nan),
                    sub["okx_freshness_ms"].fillna(np.nan),
                    sub["bybit_freshness_ms"].fillna(np.nan),
                ], axis=-1),
                hovertemplate=(
                    "coin=%{text}<br>"
                    "time=%{x}<br>"
                    "spread=%{y:.6f}<br>"
                    "trigger=%{customdata[0]}<br>"
                    "okx_latency_ms=%{customdata[1]:.2f}<br>"
                    "bybit_latency_ms=%{customdata[2]:.2f}<br>"
                    "okx_freshness_ms=%{customdata[3]:.2f}<br>"
                    "bybit_freshness_ms=%{customdata[4]:.2f}<extra></extra>"
                ),
                text=[legend_name] * len(sub),
            )
        )

    fig.update_layout(
        width=1400,
        height=700,
        xaxis_title="Event time",
        yaxis_title="Spread",
        hovermode="closest",
        legend=dict(font=dict(size=11)),
        margin=dict(l=50, r=20, t=60, b=40),
    )
    fig.show()
def plot_spread_3d(df: pd.DataFrame, x_unit: str = "seconds") -> None:
    if df.empty:
        raise ValueError("plot_df is empty")

    df = df.copy().sort_values(["coin_rank", "event_dt"])

    t0 = df["event_dt"].min()
    df["seconds_from_start"] = (df["event_dt"] - t0).dt.total_seconds().astype(float)
    df["hours_from_start"] = df["seconds_from_start"] / 3600.0

    if x_unit == "seconds":
        x_col = "seconds_from_start"
        x_title = "Seconds from window start"
    elif x_unit == "hours":
        x_col = "hours_from_start"
        x_title = "Hours from window start"
    else:
        raise ValueError("x_unit must be 'seconds' or 'hours'")

    print("X column:", x_col)
    print("X range:", df[x_col].min(), "->", df[x_col].max())
    print("event_dt range:", df["event_dt"].min(), "->", df["event_dt"].max())

    rank_min = df["coin_rank"].min()
    df["coin_y"] = df["coin_rank"] - rank_min

    y_tick_df = (
        df[["coin_rank", "coin_y", "legend_name"]]
        .drop_duplicates()
        .sort_values("coin_rank")
    )

    zmin = df["spread_value"].min()
    zmax = df["spread_value"].max()

    fig = go.Figure()

    for _, sub in df.groupby("coin_rank", sort=True):
        sub = sub.sort_values("event_dt")
        legend_name = sub["legend_name"].iloc[0]

        fig.add_trace(
            go.Scatter3d(
                x=sub[x_col].to_numpy(dtype=float),
                y=sub["coin_y"].to_numpy(dtype=float),
                z=sub["spread_value"].to_numpy(dtype=float),
                mode="lines+markers",
                name=legend_name,
                line=dict(width=4),
                marker=dict(size=2.8),
                opacity=0.9,
                customdata=np.stack([
                    sub["event_dt"].astype(str),
                    sub["trigger"].astype(str),
                    sub["okx_latency_ms"].fillna(np.nan).to_numpy(),
                    sub["bybit_latency_ms"].fillna(np.nan).to_numpy(),
                    sub["okx_freshness_ms"].fillna(np.nan).to_numpy(),
                    sub["bybit_freshness_ms"].fillna(np.nan).to_numpy(),
                ], axis=-1),
                hovertemplate=(
                    "coin=%{text}<br>"
                    "time=%{customdata[0]}<br>"
                    f"{x_title}=%{{x:.4f}}<br>"
                    "spread=%{z:.6f}<br>"
                    "trigger=%{customdata[1]}<br>"
                    "okx_latency_ms=%{customdata[2]:.2f}<br>"
                    "bybit_latency_ms=%{customdata[3]:.2f}<br>"
                    "okx_freshness_ms=%{customdata[4]:.2f}<br>"
                    "bybit_freshness_ms=%{customdata[5]:.2f}<extra></extra>"
                ),
                text=[legend_name] * len(sub),
            )
        )

    fig.update_layout(
        title="Spread map (3D)",
        width=1500,
        height=950,
        scene=dict(
            xaxis=dict(
                title=x_title,
                backgroundcolor="rgb(245,245,245)",
                gridcolor="lightgray",
                zerolinecolor="gray",
            ),
            yaxis=dict(
                title="Coin rank in block",
                tickmode="array",
                tickvals=y_tick_df["coin_y"].tolist(),
                ticktext=y_tick_df["legend_name"].tolist(),
                backgroundcolor="rgb(245,245,245)",
                gridcolor="lightgray",
            ),
            zaxis=dict(
                title="Spread",
                range=[zmin, zmax],
                backgroundcolor="rgb(245,245,245)",
                gridcolor="lightgray",
                zerolinecolor="black",
            ),
            aspectmode="manual",
            aspectratio=dict(x=2.6, y=1.4, z=1.1),
            camera=dict(eye=dict(x=1.8, y=1.7, z=1.15)),
        ),
        legend=dict(font=dict(size=11)),
        margin=dict(l=0, r=0, t=50, b=0),
    )
    fig.show()
def build_spread_2d_figure(df: pd.DataFrame) -> go.Figure:
    if df.empty:
        raise ValueError("plot_df is empty")

    fig = go.Figure()

    for _, sub in df.groupby("coin_rank", sort=True):
        sub = sub.sort_values("event_dt")
        legend_name = sub["legend_name"].iloc[0]

        fig.add_trace(
            go.Scatter(
                x=sub["event_dt"],
                y=sub["spread_value"],
                mode="lines+markers",
                name=legend_name,
                line=dict(width=1.7),
                marker=dict(size=3),
                customdata=np.stack([
                    sub["trigger"].astype(str),
                    sub["okx_latency_ms"].fillna(np.nan),
                    sub["bybit_latency_ms"].fillna(np.nan),
                    sub["okx_freshness_ms"].fillna(np.nan),
                    sub["bybit_freshness_ms"].fillna(np.nan),
                ], axis=-1),
                hovertemplate=(
                    "coin=%{text}<br>"
                    "time=%{x}<br>"
                    "spread=%{y:.6f}<br>"
                    "trigger=%{customdata[0]}<br>"
                    "okx_latency_ms=%{customdata[1]:.2f}<br>"
                    "bybit_latency_ms=%{customdata[2]:.2f}<br>"
                    "okx_freshness_ms=%{customdata[3]:.2f}<br>"
                    "bybit_freshness_ms=%{customdata[4]:.2f}<extra></extra>"
                ),
                text=[legend_name] * len(sub),
            )
        )

    fig.update_layout(
        title="2D spread",
        width=900,
        height=520,
        xaxis_title="Event time",
        yaxis_title="Spread",
        hovermode="closest",
        legend=dict(font=dict(size=10)),
        margin=dict(l=40, r=20, t=50, b=40),
    )
    return fig


def build_spread_3d_figure(df: pd.DataFrame, x_unit: str = "seconds") -> go.Figure:
    if df.empty:
        raise ValueError("plot_df is empty")

    df = df.copy().sort_values(["coin_rank", "event_dt"])

    t0 = df["event_dt"].min()
    df["seconds_from_start"] = (df["event_dt"] - t0).dt.total_seconds().astype(float)
    df["hours_from_start"] = df["seconds_from_start"] / 3600.0

    if x_unit == "seconds":
        x_col = "seconds_from_start"
        x_title = "Seconds from start"
    elif x_unit == "hours":
        x_col = "hours_from_start"
        x_title = "Hours from start"
    else:
        raise ValueError("x_unit must be 'seconds' or 'hours'")

    rank_min = df["coin_rank"].min()
    df["coin_y"] = df["coin_rank"] - rank_min

    y_tick_df = (
        df[["coin_rank", "coin_y", "legend_name"]]
        .drop_duplicates()
        .sort_values("coin_rank")
    )

    zmin = df["spread_value"].min()
    zmax = df["spread_value"].max()

    fig = go.Figure()

    for _, sub in df.groupby("coin_rank", sort=True):
        sub = sub.sort_values("event_dt")
        legend_name = sub["legend_name"].iloc[0]

        fig.add_trace(
            go.Scatter3d(
                x=sub[x_col].to_numpy(dtype=float),
                y=sub["coin_y"].to_numpy(dtype=float),
                z=sub["spread_value"].to_numpy(dtype=float),
                mode="lines+markers",
                name=legend_name,
                line=dict(width=4),
                marker=dict(size=2.5),
                opacity=0.9,
                customdata=np.stack([
                    sub["event_dt"].astype(str),
                    sub["trigger"].astype(str),
                    sub["okx_latency_ms"].fillna(np.nan).to_numpy(),
                    sub["bybit_latency_ms"].fillna(np.nan).to_numpy(),
                    sub["okx_freshness_ms"].fillna(np.nan).to_numpy(),
                    sub["bybit_freshness_ms"].fillna(np.nan).to_numpy(),
                ], axis=-1),
                hovertemplate=(
                    "coin=%{text}<br>"
                    "time=%{customdata[0]}<br>"
                    f"{x_title}=%{{x:.4f}}<br>"
                    "spread=%{z:.6f}<br>"
                    "trigger=%{customdata[1]}<br>"
                    "okx_latency_ms=%{customdata[2]:.2f}<br>"
                    "bybit_latency_ms=%{customdata[3]:.2f}<br>"
                    "okx_freshness_ms=%{customdata[4]:.2f}<br>"
                    "bybit_freshness_ms=%{customdata[5]:.2f}<extra></extra>"
                ),
                text=[legend_name] * len(sub),
            )
        )

    fig.update_layout(
        title="3D spread map",
        width=900,
        height=520,
        scene=dict(
            xaxis=dict(title=x_title),
            yaxis=dict(
                title="Coin rank",
                tickmode="array",
                tickvals=y_tick_df["coin_y"].tolist(),
                ticktext=y_tick_df["legend_name"].tolist(),
            ),
            zaxis=dict(
                title="Spread",
                range=[zmin, zmax],
            ),
            aspectmode="manual",
            aspectratio=dict(x=2.2, y=1.3, z=1.0),
            camera=dict(eye=dict(x=1.7, y=1.55, z=1.05)),
        ),
        legend=dict(font=dict(size=10)),
        margin=dict(l=0, r=0, t=50, b=0),
    )
    return fig

def build_dashboard_html(
    start_dt: str,
    end_dt: str,
    spread_col: str = "spread_long",
    block_size: int = 30,
    max_rows_per_coin: int = 200,
    x_unit: str = "seconds",
    output_file: str = "output/spread_dashboard_1.html",
    max_blocks= None,
):
    con = duckdb.connect()
    total_coins_sql = f"""
        SELECT COUNT(DISTINCT base_coin) AS n_coins
        FROM read_parquet('{PARQUET_GLOB}')
        WHERE event_dt >= TIMESTAMP '{start_dt}'
          AND event_dt < TIMESTAMP '{end_dt}'
          AND base_coin IS NOT NULL
          AND {spread_col} IS NOT NULL
    """
    total_coins = int(con.execute(total_coins_sql).fetchone()[0])

    if total_coins == 0:
        raise ValueError("No coins found in selected window")

    offsets = list(range(0, total_coins, block_size))
    if max_blocks is not None:
        offsets = offsets[:max_blocks]

    sections = []
    plotly_js_included = False

    for block_idx, offset in enumerate(offsets, start=1):
        df_block = load_plot_df(
            start_dt=start_dt,
            end_dt=end_dt,
            spread_col=spread_col,
            coin_offset=offset,
            top_n_coins=block_size,
            max_rows_per_coin=max_rows_per_coin,
        )

        if df_block.empty:
            continue

        tmin = df_block["event_dt"].min()
        tmax = df_block["event_dt"].max()
        dur_sec = (tmax - tmin).total_seconds()
        coin_count = df_block["base_coin"].nunique()
        row_count = len(df_block)

        fig3d = build_spread_3d_figure(df_block, x_unit=x_unit)
        fig2d = build_spread_2d_figure(df_block)

        div3d = plot(
            fig3d,
            include_plotlyjs=("cdn" if not plotly_js_included else False),
            output_type="div",
        )
        plotly_js_included = True

        div2d = plot(
            fig2d,
            include_plotlyjs=False,
            output_type="div",
        )

        section_html = f"""
        <section class="block">
            <div class="block-header">
                <h2>Block {block_idx}: coins {offset + 1}–{offset + coin_count}</h2>
                <div class="meta">
                    rows={row_count} | coins={coin_count} | time={tmin} → {tmax} | duration_sec={dur_sec:.2f}
                </div>
            </div>
            <div class="grid">
                <div class="panel left">
                    {div3d}
                </div>
                <div class="panel right">
                    {div2d}
                </div>
            </div>
        </section>
        """
        sections.append(section_html)
        print(f"built block={block_idx} offset={offset} rows={row_count} coins={coin_count}")

    html = f"""
    <!doctype html>
    <html lang="en">
    <head>
        <meta charset="utf-8">
        <title>Spread dashboard</title>
        <style>
            body {{
                margin: 0;
                font-family: Arial, sans-serif;
                background: #f5f5f5;
                color: #111;
            }}
            .page {{
                width: 100%;
                box-sizing: border-box;
                padding: 24px 20px 80px 20px;
            }}
            h1 {{
                margin: 0 0 8px 0;
                font-size: 28px;
            }}
            .sub {{
                margin-bottom: 24px;
                color: #555;
                font-size: 14px;
            }}
            .block {{
                background: white;
                border-radius: 12px;
                padding: 18px 18px 10px 18px;
                margin-bottom: 26px;
                box-shadow: 0 2px 10px rgba(0,0,0,0.06);
            }}
            .block-header {{
                margin-bottom: 12px;
            }}
            .block-header h2 {{
                margin: 0 0 6px 0;
                font-size: 20px;
            }}
            .meta {{
                font-size: 13px;
                color: #666;
            }}
            .grid {{
                display: grid;
                grid-template-columns: 1fr 1fr;
                gap: 18px;
                align-items: start;
            }}
            .panel {{
                min-width: 0;
                overflow: hidden;
            }}
            @media (max-width: 1400px) {{
                .grid {{
                    grid-template-columns: 1fr;
                }}
            }}
        </style>
    </head>
    <body>
        <div class="page">
            <h1>Spread dashboard</h1>
            <div class="sub">
                window={start_dt} → {end_dt} | spread_col={spread_col} | block_size={block_size} | max_rows_per_coin={max_rows_per_coin}
            </div>
            {''.join(sections)}
        </div>
    </body>
    </html>
    """

    output_path = Path(output_file)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(html, encoding="utf-8")
    print(f"saved html: {output_path.resolve()}")
    return output_path

dashboard_path = build_dashboard_html(
    start_dt="2026-07-20 21:00:00",
    end_dt="2026-07-21 10:00:00",
    spread_col="spread_short",
    block_size=5,
    max_rows_per_coin=500,
    x_unit="seconds",
    output_file="output/spread_dashboard_1.html",
    max_blocks=80,   
)

print(dashboard_path)

## Плот всех меток шорт и лонг спреда определенной монеты в фиксированном временном промежутке

In [ ]:
def load_coin_dual_spread_df(
    start_dt,
    end_dt,
    base_coin,
):
    con = duckdb.connect()

    query_sql = f"""
        SELECT
            event_dt,
            base_coin,
            trigger,
            spread_short,
            spread_long,
            okx_latency_ms,
            bybit_latency_ms,
            okx_freshness_ms,
            bybit_freshness_ms,
            max_latency_ms,
            max_freshness_ms
        FROM read_parquet('{PARQUET_GLOB}')
        WHERE event_dt >= TIMESTAMP '{start_dt}'
          AND event_dt < TIMESTAMP '{end_dt}'
          AND base_coin = '{base_coin}'
          AND (
                spread_short IS NOT NULL
                OR spread_long IS NOT NULL
              )
        ORDER BY event_dt
    """

    df = con.execute(query_sql).df()

    if df.empty:
        return df

    df["event_dt"] = pd.to_datetime(df["event_dt"])
    return df.reset_index(drop=True)

def inspect_coin_dual_spread_df(df):
    if df.empty:
        print("coin_df is empty")
        return

    print("shape:", df.shape)
    print("coin:", df["base_coin"].iloc[0])
    print("time min:", df["event_dt"].min())
    print("time max:", df["event_dt"].max())
    print("duration sec:", (df["event_dt"].max() - df["event_dt"].min()).total_seconds())

    print("spread_short non-null:", df["spread_short"].notna().sum())
    print("spread_long  non-null:", df["spread_long"].notna().sum())

    if df["spread_short"].notna().any():
        print("spread_short min:", df["spread_short"].min())
        print("spread_short max:", df["spread_short"].max())

    if df["spread_long"].notna().any():
        print("spread_long min:", df["spread_long"].min())
        print("spread_long max:", df["spread_long"].max())

    print("trigger counts:")
    print(df["trigger"].value_counts(dropna=False))

def plot_coin_short_long(df, title=None):
    if df.empty:
        raise ValueError("coin_df is empty")

    base_coin = df["base_coin"].iloc[0]
    df = df.sort_values("event_dt").copy()

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=df["event_dt"],
            y=df["spread_short"],
            mode="lines+markers",
            name="spread_short",
            line=dict(width=1.6, color="#d62728"),
            marker=dict(size=3),
            customdata=np.stack([
                df["trigger"].astype(str),
                df["okx_latency_ms"].fillna(np.nan),
                df["bybit_latency_ms"].fillna(np.nan),
                df["okx_freshness_ms"].fillna(np.nan),
                df["bybit_freshness_ms"].fillna(np.nan),
                df["max_latency_ms"].fillna(np.nan),
                df["max_freshness_ms"].fillna(np.nan),
            ], axis=-1),
            hovertemplate=(
                f"coin={base_coin}<br>"
                "series=spread_short<br>"
                "time=%{x}<br>"
                "value=%{y:.6f}<br>"
                "trigger=%{customdata[0]}<br>"
                "okx_latency_ms=%{customdata[1]:.2f}<br>"
                "bybit_latency_ms=%{customdata[2]:.2f}<br>"
                "okx_freshness_ms=%{customdata[3]:.2f}<br>"
                "bybit_freshness_ms=%{customdata[4]:.2f}<br>"
                "max_latency_ms=%{customdata[5]:.2f}<br>"
                "max_freshness_ms=%{customdata[6]:.2f}<extra></extra>"
            ),
            connectgaps=False,
        )
    )

    fig.add_trace(
        go.Scatter(
            x=df["event_dt"],
            y=df["spread_long"],
            mode="lines+markers",
            name="spread_long",
            line=dict(width=1.6, color="#1f77b4"),
            marker=dict(size=3),
            customdata=np.stack([
                df["trigger"].astype(str),
                df["okx_latency_ms"].fillna(np.nan),
                df["bybit_latency_ms"].fillna(np.nan),
                df["okx_freshness_ms"].fillna(np.nan),
                df["bybit_freshness_ms"].fillna(np.nan),
                df["max_latency_ms"].fillna(np.nan),
                df["max_freshness_ms"].fillna(np.nan),
            ], axis=-1),
            hovertemplate=(
                f"coin={base_coin}<br>"
                "series=spread_long<br>"
                "time=%{x}<br>"
                "value=%{y:.6f}<br>"
                "trigger=%{customdata[0]}<br>"
                "okx_latency_ms=%{customdata[1]:.2f}<br>"
                "bybit_latency_ms=%{customdata[2]:.2f}<br>"
                "okx_freshness_ms=%{customdata[3]:.2f}<br>"
                "bybit_freshness_ms=%{customdata[4]:.2f}<br>"
                "max_latency_ms=%{customdata[5]:.2f}<br>"
                "max_freshness_ms=%{customdata[6]:.2f}<extra></extra>"
            ),
            connectgaps=False,
        )
    )

    fig.update_layout(
        title=title or f"{base_coin}: spread_short + spread_long",
        width=1500,
        height=750,
        xaxis_title="Event time",
        yaxis_title="Spread",
        hovermode="x unified",
        legend=dict(font=dict(size=11)),
        margin=dict(l=50, r=20, t=60, b=40),
    )
    fig.show()

coin_df = load_coin_dual_spread_df(
    start_dt="2026-07-21 2:01:40",
    end_dt="2026-07-21 7:05:00",
    base_coin="LA",
)

inspect_coin_dual_spread_df(coin_df)
plot_coin_short_long(coin_df)